In [1]:
!git clone https://github.com/Brayan695/RNAseq-AMD.git

Cloning into 'RNAseq-AMD'...
remote: Enumerating objects: 2771, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 2771 (delta 13), reused 0 (delta 0), pack-reused 2746 (from 2)
Receiving objects: 100% (2771/2771), 1.05 GiB | 20.13 MiB/s, done.
Resolving deltas: 100% (1022/1022), done.
Updating files: 100% (1876/1876), done.


In [2]:
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.feature_selection import f_classif
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [3]:
DATA_PATH = "/kaggle/working/RNAseq-AMD/Dataset/MetaSheet_Processed.csv"
N_ITERATIONS = 1000
SAMPLE_FRACTION = 0.8
TOP_K_FRACTION = 0.30     #per iteration: keep top 30% of usable features by score
TOP_N_FRACTION = 0.10     #final: keep top 10% of usable features by selection frequency                                      

MIN_METHODS_AGREEMENT = 2
# A feature is kept if it lands in the top-N for at least this many of the
# 3 methods (anova/auc/kruskal), not necessarily all 3. Set to 3 to restore
# the original strict-intersection behavior. Relaxing this
# to 2-of-3 is a deliberate experiment to compare against the strict version
# (smoking survived variance checks but never cleared the strict 3/3 bar).
RANDOM_SEED = 2026

# All 6 pairwise MGS stage comparisons to run. Each tuple is (negative_class_stage, positive_class_stage) 
# for ex: (1, 4) means "control (MGS1) vs. late AMD (MGS4)"
STAGE_PAIRS = [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]

EXCLUDE_LABEL_LEAKAGE = True
LEAKAGE_COLUMNS = [
    "oc_AMD",
    "oc_dry AMD",
    "oc_macular degeneration",
    "oc_macular degeneraton",
    "oc_AMD (Avastin injections)",
    "oc_AMD (Lucentis injections)",
    "oc_AMD (OD)",
    "oc_on Ocuvite"
    "oc_AMD (took vitamins)",
    "oc_AMD (wet)",
    "oc_AMD -took vitamins",
    "oc_retinal injection",
    "oc_AMD?",
    "oc_Wet AMD",
    "oc_wet AMD",
    "oc_early AMD",
    "oc_possible AMD",
    "oc_possible macular degeneration",
    "oc_AMD (received shots)",
]

COLLAPSE_MAP_PATH = "/kaggle/working/RNAseq-AMD/Dataset/MetaSheet_collapsed_with_flags_mapping.csv"


EXCLUDE_CONFOUNDS = True
CONFOUND_COLUMNS = ["age"]

EXCLUDE_NO_INFO = True
NO_INFO_COLUMNS = ["mh_-", "mh_N/A", "oc_?", "oc_n/a", "oc_unknown", "0", "oc_Saw eye doctor","oc_drops at time of surgery"]

rng = np.random.default_rng(RANDOM_SEED)

In [4]:
def load_data(path, stage_pair=(1, 4)):
    """
    Load metadata, filter to two MGS stages, split into features (X) and
    label (y). y = 1 for the higher numbered (more advanced) stage in
    stage_pair, 0 for the lower numbered stage. All other stages are
    excluded from this comparison entirely.
    """
    neg_stage, pos_stage = stage_pair
    df = pd.read_csv(path)
    df = df[df["mgs_level"].isin([neg_stage, pos_stage])].reset_index(drop=True)
    y = (df["mgs_level"] == pos_stage).astype(int).values
    drop_cols = [c for c in ["sample_id", "mgs_level"] if c in df.columns]
    X = df.drop(columns=drop_cols)
    return X, y

In [5]:
def drop_constant_features(X):
    """Remove features with zero variance"""
    nunique = X.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()
    X_filtered = X.drop(columns=constant_cols)
    return X_filtered, constant_cols

## Column review: full list after dropping leakage/confound/zero-variance columns

Run this once, on the full (all-stage) dataset, to get a single alphabetized column list to manually scan for near-duplicate features before consolidating and re-running feature selection.

In [6]:
# --- Full column list for manual review

df_full = pd.read_csv(DATA_PATH)
drop_cols = [c for c in ["sample_id", "mgs_level"] if c in df_full.columns]
X_full = df_full.drop(columns=drop_cols)

if EXCLUDE_LABEL_LEAKAGE:
    present = [c for c in LEAKAGE_COLUMNS if c in X_full.columns]
    X_full = X_full.drop(columns=present)
    print(f"dropped {len(present)} label-leakage columns")

if EXCLUDE_CONFOUNDS:
    present = [c for c in CONFOUND_COLUMNS if c in X_full.columns]
    X_full = X_full.drop(columns=present)
    print(f"dropped {len(present)} confound columns")

if EXCLUDE_NO_INFO:
    present = [c for c in NO_INFO_COLUMNS if c in X_full.columns]
    X_full = X_full.drop(columns=present)
    print(f"dropped {len(present)} no-info/placeholder columns: {present}")

X_full_nonconstant, dropped_full = drop_constant_features(X_full)

print(f"\nfull dataset (all 4 stages): {X_full.shape[1]} columns after leakage/confound drop")
print(f"dropped {len(dropped_full)} constant (zero-variance) columns")
print(f"{X_full_nonconstant.shape[1]} usable columns remain\n")

remaining_cols = sorted(X_full_nonconstant.columns.tolist())
for c in remaining_cols:
    print(c)

pd.Series(remaining_cols, name="column").to_csv(
    "metadata_columns_for_review.csv", index=False
)
print(f"\nSaved: metadata_columns_for_review.csv ({len(remaining_cols)} columns)")

dropped 17 label-leakage columns
dropped 1 confound columns
dropped 7 no-info/placeholder columns: ['mh_-', 'mh_N/A', 'oc_?', 'oc_n/a', 'oc_unknown', 'oc_Saw eye doctor', 'oc_drops at time of surgery']

full dataset (all 4 stages): 589 columns after leakage/confound drop
dropped 0 constant (zero-variance) columns
589 usable columns remain

A69S_GG
A69S_GT
A69S_TT
Y402H_CC
Y402H_CT
Y402H_TT
mh_1/2 pack day for 60 yrs; HBP
mh_A Fib
mh_A fib
mh_A. Fib
mh_A. fib
mh_A. fib (pacemaker) - 2007; high cholesterol)
mh_AA repair
mh_AAA
mh_AML
mh_ARDS
mh_Acute Renal Failure
mh_Afib
mh_Alcohol abuse
mh_Alcoholic Cirrhosis
mh_Alzheimer's
mh_Alzheimer's depression
mh_Alzheimers
mh_Anemia
mh_Aneurysm
mh_Bladder cancer with mets
mh_Breast cancer
mh_C.diff
mh_CABG
mh_CABG - 1979
mh_CAD
mh_CAD (CABG - 1998)
mh_CAD s/p CABG
mh_CAD s/p stent
mh_CHF
mh_CKD
mh_CKD - stage 3
mh_CLL
mh_CMF+Afib post op
mh_CML
mh_COPD
mh_CVA
mh_Cardiomyopathy
mh_Cerebrovascular accident
mh_Chronic Kidney disease
mh_Coronary Art

## Collapse near-duplicate columns

Loads the annotated `Feature: / Flag: / Collapse Group: / Notes:` CSV produced from the manual
review above. Any row with a non-empty `Collapse Group:` gets merged into one column with its
siblings; every source column in a group should represent the identical clinical fact for a
given patient, so the merge rule is **logical OR** (`max` across 0/1 columns) -- if any of the
duplicate-spelling columns is 1 for a patient, the collapsed column is 1.

Rows with an empty `Collapse Group:` (including anything flagged as label leakage) pass through
unchanged -- they either don't have a literal duplicate, or they're excluded upstream by
`LEAKAGE_COLUMNS` already.

In [7]:
def load_collapse_mapping(path):
    """Build a raw-column -> list-of-canonical-names mapping from the annotated
    review CSV. Rows with a non-empty 'Collapse Group:' map to that group (their
    merge target); a compound entry lists MULTIPLE targets separated by '; '
    (e.g. a "smoker; HTN" cell maps to ["Smoking history...", "Hypertension (HTN)"]).
    Rows with an empty Collapse Group map to a single-item list containing just
    themselves (no merge). Every raw column therefore maps to a list, even when
    that list has only one entry, so the collapse step below can treat ordinary
    duplicates and compound fan-out with the same logic."""
    review = pd.read_csv(path)
    mapping = {}
    for _, row in review.iterrows():
        feat = row["Feature:"]
        collapse = row.get("Collapse Group:", np.nan)
        if pd.isna(collapse) or str(collapse).strip() == "":
            mapping[feat] = [feat]
        else:
            # Split on ';' for compound entries. A raw column listed under
            # several canonical targets contributes its value to EVERY one of
            # them (fan-out) -- if the compound cell was 1 for a patient, each
            # split-out target column is 1 for that same patient, same as if
            # the raw data had separate columns for each condition to begin with.
            targets = [t.strip() for t in str(collapse).split(";") if t.strip()]
            mapping[feat] = targets
    return mapping


def collapse_duplicate_columns(X, mapping, verbose=True):
    """Merge near-duplicate columns in X per `mapping` (raw col -> list of
    canonical names). Multiple raw columns landing on the same canonical name
    are combined via logical OR (elementwise max on the 0/1 columns) -- this
    covers both ordinary duplicates (many raw cols -> one canonical) and
    compound entries (one raw col -> many canonicals, fanned out here so each
    target canonical still gets OR-merged with any other raw columns that also
    map to it). Columns not present in `mapping` at all (e.g. genotype/sex_bin,
    or anything not in the review CSV) pass through untouched under their
    original name."""
    canonical_to_raw = {}
    n_fanout_cols = 0
    for col in X.columns:
        targets = mapping.get(col, [col])
        if len(targets) > 1:
            n_fanout_cols += 1
        for canonical in targets:
            canonical_to_raw.setdefault(canonical, []).append(col)

    merged_data = {}
    n_merged_groups = 0
    n_cols_absorbed = 0
    for canonical, raw_cols in canonical_to_raw.items():
        if len(raw_cols) == 1:
            merged_data[canonical] = X[raw_cols[0]]
        else:
            merged_data[canonical] = X[raw_cols].max(axis=1)
            n_merged_groups += 1
            n_cols_absorbed += len(raw_cols) - 1

    X_collapsed = pd.DataFrame(merged_data, index=X.index)
    if verbose:
        msg = (f"  collapsed {n_merged_groups} groups, absorbing {n_cols_absorbed} "
               f"duplicate columns ({X.shape[1]} -> {X_collapsed.shape[1]} columns)")
        if n_fanout_cols:
            msg += f"; {n_fanout_cols} compound column(s) fanned out to multiple targets"
        print(msg)
    return X_collapsed


COLLAPSE_MAPPING = load_collapse_mapping(COLLAPSE_MAP_PATH)
_all_targets = set(t for targets in COLLAPSE_MAPPING.values() for t in targets)
_n_compound = sum(1 for targets in COLLAPSE_MAPPING.values() if len(targets) > 1)
print(f"Loaded collapse mapping for {len(COLLAPSE_MAPPING)} reviewed columns "
      f"({len(_all_targets)} distinct canonical targets, {_n_compound} compound "
      f"entries feeding multiple targets)")

Loaded collapse mapping for 604 reviewed columns (175 distinct canonical targets, 524 compound entries feeding multiple targets)


In [8]:
# --- Distinct flag/collapse-group count + frequency table -----------------

from collections import Counter

target_counts = Counter(
    target
    for targets in COLLAPSE_MAPPING.values()
    for target in targets
)

freq_table = (
    pd.Series(target_counts, name="n_features")
    .rename_axis("collapse_group")
    .reset_index()
    .sort_values("n_features", ascending=False)
    .reset_index(drop=True)
)

print(f"Distinct collapse groups / flags: {freq_table.shape[0]}")
print(freq_table.to_string(index=False))

freq_table.to_csv("collapse_group_frequency_table.csv", index=False)
print("\nSaved: collapse_group_frequency_table.csv")

Distinct collapse groups / flags: 175
                           collapse_group  n_features
               chronic inflammatory issue         424
                          Smoking history         149
                 acute inflammatory issue          80
                         autoimmune issue          29
                                Cataracts          25
                     general inflammation          20
                       Hypertension (HTN)          19
                                        0          17
        High cholesterol / hyperlipidemia          16
            Coronary artery disease (CAD)          15
                             Pseudophakic          14
                            Heart disease          14
                  Kidney disease/problems          12
                       Retinal detachment          11
              Atrial fibrillation (A Fib)          10
                Rheumatoid arthritis (RA)           9
                                  Dry eye   

In [9]:
def anova_scores(X, y):
    """ANOVA F-test score per feature, all features at once."""
    f_stat, _ = f_classif(X.values, y)
    f_stat = np.nan_to_num(f_stat, nan=0.0)
    return pd.Series(f_stat, index=X.columns)
 
def auc_scores(X, y):
    """AUC per feature, computed directly from ranks (equivalent to
    roc_auc_score but vectorized across every column at once)
    Take max(auc, 1-auc) so the direction of the association (e.g.
    presence vs. absence of a condition) doesn't penalize the score."""
    n1 = y.sum()
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return pd.Series(0.5, index=X.columns)
    ranks = stats.rankdata(X.values, axis=0, method="average")
    sum_ranks_pos = ranks[y == 1].sum(axis=0)
    auc = (sum_ranks_pos - n1 * (n1 + 1) / 2) / (n1 * n0)
    auc = np.maximum(auc, 1 - auc)
    return pd.Series(auc, index=X.columns)
 
def kruskal_scores(X, y):
    """Kruskal Wallis H statistic per feature  computed directly from ranks"""
    N = len(y)
    ranks = stats.rankdata(X.values, axis=0, method="average")
    n1 = y.sum()
    n0 = N - n1
    R1 = ranks[y == 1].sum(axis=0)
    R0 = ranks[y == 0].sum(axis=0)
    H = (12 / (N * (N + 1))) * ((R1 ** 2) / n1 + (R0 ** 2) / n0) - 3 * (N + 1)
    # Tie correction: C = 1 - sum(t^3 - t) / (N^3 - N) computed per column
    tie_correction = np.ones(X.shape[1])
    X_vals = X.values
    for i in range(X.shape[1]):
        _, counts = np.unique(X_vals[:, i], return_counts=True)
        tie_sum = np.sum(counts ** 3 - counts)
        tie_correction[i] = 1 - tie_sum / (N ** 3 - N)
    with np.errstate(divide="ignore", invalid="ignore"):
        H_corrected = np.where(tie_correction > 0, H / tie_correction, 0.0)
    H_corrected = np.nan_to_num(H_corrected, nan=0.0, posinf=0.0, neginf=0.0)
    return pd.Series(H_corrected, index=X.columns)

In [10]:
def run_pipeline(X, y, n_iterations, sample_fraction, top_k, seed):
    """Run the 1000 iteration resampling + scoring loop.
    Returns a DataFrame of selection frequency (0-1) per feature per method.
    """
    counts = {
        "anova": pd.Series(0, index=X.columns, dtype=int),
        "auc": pd.Series(0, index=X.columns, dtype=int),
        "kruskal": pd.Series(0, index=X.columns, dtype=int),
    }
    for it in range(n_iterations):
        #Stratified 80% resample: preserves the control to AMD ratio in
        # every iteration so small class features aren't starved.
        X_sub, _, y_sub, _ = train_test_split(
            X, y,
            train_size=sample_fraction,
            stratify=y,
            random_state=RANDOM_SEED + it,
        )
        scores = {
            "anova": anova_scores(X_sub, y_sub),
            "auc": auc_scores(X_sub, y_sub),
            "kruskal": kruskal_scores(X_sub, y_sub),
        }
        for method, s in scores.items():
            top_features = s.sort_values(ascending=False).head(top_k).index
            counts[method].loc[top_features] += 1
        if (it + 1) % 100 == 0:
            print(f"  iteration {it + 1}/{n_iterations} done")
    freq = pd.DataFrame({m: c / n_iterations for m, c in counts.items()})
    return freq

In [11]:
def select_consistent_features(freq_df, top_n, min_methods=MIN_METHODS_AGREEMENT):
    """Take the top N features per method (by selection frequency), then keep
    any feature that lands in the top N for at least `min_methods` of the 3
    methods -- NOT necessarily all 3. With min_methods=3 this reduces exactly
    to the original strict intersection; min_methods=2 is the relaxed
    "2-of-3 agreement" criterion Dr. Barman asked to try."""
    top_sets = {}
    for method in freq_df.columns:
        top_sets[method] = set(
            freq_df[method].sort_values(ascending=False).head(top_n).index
        )
    all_candidates = set().union(*top_sets.values())
    method_counts = {
        f: sum(f in top_sets[m] for m in top_sets) for f in all_candidates
    }
    consistent = {f for f, c in method_counts.items() if c >= min_methods}
    return consistent, top_sets, method_counts

In [12]:
all_results = {}
 
for stage_pair in STAGE_PAIRS:
    neg_stage, pos_stage = stage_pair
    label = f"{neg_stage}v{pos_stage}"
    print(f"\n{'='*60}\nStage comparison: MGS{neg_stage} vs. MGS{pos_stage}\n{'='*60}")
 
    X_raw, y = load_data(DATA_PATH, stage_pair=stage_pair)
    print(f"  {X_raw.shape[0]} samples, {X_raw.shape[1]} raw features")
    print(f"  class balance: {sum(y==0)} MGS{neg_stage}, {sum(y==1)} MGS{pos_stage}")
 
    if EXCLUDE_LABEL_LEAKAGE:
        present = [c for c in LEAKAGE_COLUMNS if c in X_raw.columns]
        X_raw = X_raw.drop(columns=present)
        print(f"  dropped {len(present)} label-leakage columns (chart notes "
              f"that restate the AMD diagnosis): {present}")
 
    if EXCLUDE_CONFOUNDS:
        present = [c for c in CONFOUND_COLUMNS if c in X_raw.columns]
        X_raw = X_raw.drop(columns=present)
        print(f"  dropped {len(present)} confound columns: {present}")
 
    if EXCLUDE_NO_INFO:
        present = [c for c in NO_INFO_COLUMNS if c in X_raw.columns]
        X_raw = X_raw.drop(columns=present)
        print(f"  dropped {len(present)} no-info/placeholder columns: {present}")
 
    X_raw = collapse_duplicate_columns(X_raw, COLLAPSE_MAPPING)
 
    X, dropped = drop_constant_features(X_raw)
    print(f"  dropped {len(dropped)} constant (zero-variance) features")
    print(f"  {X.shape[1]} usable features remain")
 
    top_k = max(1, int(X.shape[1] * TOP_K_FRACTION))
    top_n = max(1, int(X.shape[1] * TOP_N_FRACTION))
    print(f"  per-iteration top-K = {top_k}, final top-N per method = {top_n}")
 
    freq_df = run_pipeline(X, y, N_ITERATIONS, SAMPLE_FRACTION, top_k, RANDOM_SEED)
    consistent, top_sets, method_counts = select_consistent_features(
        freq_df, top_n, min_methods=MIN_METHODS_AGREEMENT
    )
 
    print(f"\n  Features selected by at least {MIN_METHODS_AGREEMENT} of 3 methods: {len(consistent)}")
    for f in sorted(consistent):
        print(f"    {f}  (anova={freq_df.loc[f,'anova']:.2f}, "
              f"auc={freq_df.loc[f,'auc']:.2f}, kruskal={freq_df.loc[f,'kruskal']:.2f}, "
              f"methods_agreeing={method_counts[f]}/3)")
 
    freq_suffix = f"_min{MIN_METHODS_AGREEMENT}of3"
    freq_df.to_csv(f"metadata_feature_selection_frequencies_{label}{freq_suffix}.csv")
    pd.Series(sorted(consistent), name="feature").to_csv(
        f"metadata_selected_features_{label}{freq_suffix}.csv", index=False
    )
    print(f"  Saved: metadata_feature_selection_frequencies_{label}{freq_suffix}.csv, "
          f"metadata_selected_features_{label}{freq_suffix}.csv")
 
    all_results[label] = consistent


Stage comparison: MGS1 vs. MGS2
  280 samples, 614 raw features
  class balance: 105 MGS1, 175 MGS2
  dropped 17 label-leakage columns (chart notes that restate the AMD diagnosis): ['oc_AMD', 'oc_dry AMD', 'oc_macular degeneration', 'oc_macular degeneraton', 'oc_AMD (Avastin injections)', 'oc_AMD (Lucentis injections)', 'oc_AMD (OD)', 'oc_AMD (wet)', 'oc_AMD -took vitamins', 'oc_retinal injection', 'oc_AMD?', 'oc_Wet AMD', 'oc_wet AMD', 'oc_early AMD', 'oc_possible AMD', 'oc_possible macular degeneration', 'oc_AMD (received shots)']
  dropped 1 confound columns: ['age']
  dropped 7 no-info/placeholder columns: ['mh_-', 'mh_N/A', 'oc_?', 'oc_n/a', 'oc_unknown', 'oc_Saw eye doctor', 'oc_drops at time of surgery']
  collapsed 83 groups, absorbing 997 duplicate columns (589 -> 175 columns); 524 compound column(s) fanned out to multiple targets
  dropped 26 constant (zero-variance) features
  149 usable features remain
  per-iteration top-K = 44, final top-N per method = 14
  iteration 100

In [13]:

#which features are robust across MULTIPLE stage comparisons not just one?

all_features = sorted(set().union(*all_results.values()))
summary = pd.DataFrame(index=all_features)
for label, feats in all_results.items():
    summary[label] = summary.index.isin(feats)
summary["n_pairs_selected"] = summary.sum(axis=1)
summary = summary.sort_values("n_pairs_selected", ascending=False)
 
print(f"\n{'='*60}\nCross-pair summary\n{'='*60}")
print(summary)
summary.to_csv("metadata_feature_selection_cross_pair_summary.csv")
print("\nSaved: metadata_feature_selection_cross_pair_summary.csv")


Cross-pair summary
                                     1v2    1v3    1v4    2v3    2v4    3v4  \
Cataracts                           True   True   True   True   True  False   
A69S_GG                            False  False   True   True   True   True   
Pseudophakic                       False   True   True   True   True  False   
Alzheimer's disease                 True   True  False  False   True   True   
Sepsis                             False   True   True   True   True  False   
Coronary artery disease (CAD)       True   True  False   True  False  False   
Alcohol abuse                      False   True  False   True   True  False   
A69S_TT                            False  False   True  False   True   True   
Y402H_TT                           False  False   True   True   True  False   
Y402H_CC                           False  False   True   True   True  False   
Parkinson's disease                False  False  False   True   True  False   
A69S_GT                         